# Plasma-Column Analysis Plots

Auto-discovers completed runs under `runs/`, loads diagnostics,
and generates the full publication figure set.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RESULTS_DIR    = _ROOT / 'results'
RUNS_DIR       = _ROOT / 'results'
PLOTS_DIR      = _ROOT / 'plots'
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
_DEFAULTS = {
    'runs root':               str(RUNS_DIR),
    'figures root':            str(PLOTS_DIR),
    'expected methods':        'vacuum, seeded, callback',
    'output format':           'PNG (300 DPI) + PDF',
    'beam energy [keV]':       30.0,
    'beam current [mA]':       10.0,
    'plasma cell length [m]':  0.20,
    'inflector aperture [mm]': 5.0,
}
print_simulation_config(
    notebook_title='Plasma-Column Analysis Plots',
    defaults=_DEFAULTS, overrides={},
)


## 1. Discover and load cases


In [ ]:
from plasma_column.plotting import (
    setup_publication_style,
    plot_multi_case_neutralization,
    plot_neutralization_evolution,
    plot_particle_counts,
    plot_keff_over_k0,
    plot_species_growth_rates,
    plot_neutralization_panel,
    plot_bunched_beam_keff,
    plot_keff_pressure_scan,
    plot_radial_density_profile,
    plot_neutralization_vs_z,
    plot_phase_space,
    save_figure,
)
from plasma_column.diagnostics import (
    load_particle_number_diagnostic,
    compute_particle_number_metrics,
    DataLoader,
)
import warnings
setup_publication_style()
print('Plotting helpers loaded.')


In [ ]:
import warnings

def _infer_meta(d):
    n = d.name.lower()
    return {
        'case':   d.name,
        'method': ('seeded'   if 'seeded'   in n else
                   'callback' if 'callback' in n else
                   'vacuum'   if 'vacuum'   in n else 'unknown'),
        'gas':    'Kr' if 'kr' in n else 'H2',
    }

def _find_diag(d):
    for p in [d / 'reducedfiles' / 'ParticleNumber_red.txt',
               d / 'neutralization_from_particle_number.csv']:
        if p.exists(): return p
    return None

cases = []
for case_dir in sorted(RUNS_DIR.iterdir()):
    if not case_dir.is_dir(): continue
    diag = _find_diag(case_dir)
    if diag is None: continue
    meta = _infer_meta(case_dir)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        hist = load_particle_number_diagnostic(diag)
        hist = compute_particle_number_metrics(hist)
    cases.append((case_dir, hist, meta))
    print(f"  {case_dir.name}  ({meta['method']} | {meta['gas']}) — {len(hist)} steps")
print(f'\nTotal cases loaded: {len(cases)}')


## 2. Multi-case neutralisation overlay


In [ ]:
multi_pairs = [(f"{m['case']} | {m['method']} | {m['gas']}", h)
               for _, h, m in cases]
for col, ylabel, oname in [
    ('eta_net',           r'$(N_e-N_i)/N_p$',   'all_eta_net'),
    ('eta_electron_only', r'$N_e/N_p$',           'all_eta_electron'),
    ('keff_over_k0',      r'$K_{\rm eff}/K_0$',  'all_keff'),
]:
    if not any(col in h.columns for _, h, _ in cases): continue
    p, _ = plot_multi_case_neutralization(
        multi_pairs, PLOTS_DIR, column=col,
        ylabel=ylabel, title=f'All cases — {ylabel}', output_name=oname,
    )
    print('Saved:', p.name)
plt.show()


## 3. Per-case 3-panel summary


In [ ]:
for _, hist, meta in cases:
    p, _ = plot_neutralization_panel(hist, PLOTS_DIR, case_name=meta['case'])
    print('Saved:', p.name)
plt.show()


## 4. Species growth rates


In [ ]:
for _, hist, meta in cases:
    if not {'Ne','Ni'}.issubset(hist.columns): continue
    p, _ = plot_species_growth_rates(hist, PLOTS_DIR,
                                     case_name=meta['case'], smooth_window=7)
    print('Saved:', p.name)
plt.show()


## 5. Bunched-beam perveance


In [ ]:
for _, hist, meta in cases:
    if 'eta_net' not in hist.columns: continue
    p, _ = plot_bunched_beam_keff(
        hist['time'].values * 1e9,
        hist['eta_net'].values.clip(0, 1), PLOTS_DIR,
        case_name=meta['case'],
        bunching_factors=[1.0, 2.0, 3.0, 5.0, 8.0],
    )
    print('Saved:', p.name)
plt.show()


## 6. Final-value summary table


In [ ]:
rows = []
for _, hist, meta in cases:
    row = dict(meta)
    for col in ['eta_electron_only', 'eta_net', 'keff_over_k0']:
        row[f'final_{col}'] = hist[col].iloc[-1] if col in hist.columns else float('nan')
    row['n_steps'] = len(hist)
    rows.append(row)
display(pd.DataFrame(rows))
